# 01: Natural Language Processing: Regex, Tokenization, Lemmatization & NER

**Track 08: Classical NLP & Word Embeddings** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Build end-to-end NLP preprocessing pipelines: Regex cleaning, stopword removal, lemmatization, Part-of-Speech (POS) tagging, and Named Entity Recognition (NER).


## 1. Text Corpus Ingestion & Tokenization Pipeline
Ingest unstructured news text data and clean whitespace, HTML tags, and punctuation.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

df = load_dataset("news_articles")
print(f"Loaded {len(df)} news articles across categories: {df['category'] if 'category' in df.columns else df['Category'].unique()}")

def clean_text(text):
    text = re.sub(r"<.*?>", "", str(text)) # Remove HTML
    text = re.sub(r"[^a-zA-Z\s]", "", text) # Keep alphabetic
    text = text.lower().strip()
    return text

text_col = "headline" if "headline" in df.columns else df.columns[1]
df["Clean_Text"] = df[text_col].apply(clean_text)
print("Sample cleaned text snippet:")
print(df["Clean_Text"].iloc[0][:200])

## 2. TF-IDF Representation & Category Keyword Extraction
Compute Term Frequency-Inverse Document Frequency:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{|D|}{|\{d \in D : t \in d\}| + 1}\right)$$

In [ ]:
tfidf = TfidfVectorizer(max_features=1000, stop_words="english", ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(df["Clean_Text"])

vocab = np.array(tfidf.get_feature_names_out())
print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")

# Extract top keywords per category
for cat in df["category"].unique():
    cat_mask = (df["category"] == cat).values
    cat_tfidf = np.asarray(X_tfidf[cat_mask].mean(axis=0)).ravel()
    top_indices = cat_tfidf.argsort()[-5:][::-1]
    top_words = vocab[top_indices]
    print(f"Top 5 Keywords for [{cat}]: {', '.join(top_words)}")